In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

np.random.seed(42)

# ============================================================
# BUILD AND EVALUATE A COMPLETE MACHINE LEARNING SYSTEM
# Example problem:
# Predict whether a customer will renew a subscription (1 = renew, 0 = not renew)
# ============================================================

# ------------------------------------------------------------
# Part 1: Learning Setup (19.1–19.2)
# ------------------------------------------------------------

# Synthetic supervised dataset
X, y = make_classification(
    n_samples=1400,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    n_repeated=0,
    n_classes=2,
    weights=[0.58, 0.42],
    class_sep=1.0,
    flip_y=0.05,
    random_state=42
)

feature_names = [
    "monthly_usage",
    "support_tickets",
    "days_since_login",
    "avg_session_time",
    "plan_level_score",
    "discount_received",
    "payment_reliability",
    "feature_adoption",
    "contract_length_score",
    "engagement_trend"
]

df = pd.DataFrame(X, columns=feature_names)
df["renewed"] = y

print("=" * 80)
print("COMPLETE MACHINE LEARNING SYSTEM")
print("=" * 80)

print("\nPART 1: LEARNING SETUP")
print("Supervised learning problem:")
print("- Goal: predict whether a customer renews a subscription.")
print("- Label: renewed (0 or 1)")
print("- Features:", feature_names)

print("\nDataset preview:")
print(df.head())

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=42
)
# Result: 60% train, 20% val, 20% test

print("\nData split sizes:")
print("Train:", X_train.shape[0])
print("Validation:", X_val.shape[0])
print("Test:", X_test.shape[0])

# Baseline model
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_val_pred = baseline.predict(X_val)

baseline_acc = accuracy_score(y_val, baseline_val_pred)
baseline_f1 = f1_score(y_val, baseline_val_pred)

print("\nBaseline model (most frequent class):")
print("Validation Accuracy:", round(baseline_acc, 4))
print("Validation F1:", round(baseline_f1, 4))

# ------------------------------------------------------------
# Part 2: Decision Tree Model (19.3)
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 2: DECISION TREE MODEL")
print("=" * 80)

tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model.fit(X_train, y_train)

tree_val_pred = tree_model.predict(X_val)
tree_acc = accuracy_score(y_val, tree_val_pred)
tree_f1 = f1_score(y_val, tree_val_pred)

print("\nDecision Tree performance:")
print("Validation Accuracy:", round(tree_acc, 4))
print("Validation F1:", round(tree_f1, 4))

print("\nInterpretable tree structure:")
tree_rules = export_text(tree_model, feature_names=feature_names)
print(tree_rules)

# ------------------------------------------------------------
# Part 3: Model Selection (19.4)
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 3: MODEL SELECTION")
print("=" * 80)

models_and_grids = {
    "DecisionTree": (
        DecisionTreeClassifier(random_state=42),
        {
            "max_depth": [3, 4, 5, 7, None],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }
    ),
    "LogisticRegression": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, random_state=42))
        ]),
        {
            "model__C": [0.01, 0.1, 1.0, 5.0, 10.0]
        }
    ),
    "KNN": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier())
        ]),
        {
            "model__n_neighbors": [3, 5, 7, 11, 15],
            "model__weights": ["uniform", "distance"]
        }
    ),
    "RandomForest": (
        RandomForestClassifier(random_state=42),
        {
            "n_estimators": [100, 200],
            "max_depth": [4, 6, 10, None],
            "min_samples_leaf": [1, 2, 4]
        }
    )
}

best_models = {}
validation_results = []

for name, (estimator, param_grid) in models_and_grids.items():
    grid = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring="f1",
        cv=5,
        n_jobs=-1
    )
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    val_pred = best_model.predict(X_val)
    val_acc = accuracy_score(y_val, val_pred)
    val_f1 = f1_score(y_val, val_pred)

    best_models[name] = best_model
    validation_results.append({
        "model": name,
        "best_params": grid.best_params_,
        "val_accuracy": val_acc,
        "val_f1": val_f1
    })

results_df = pd.DataFrame(validation_results).sort_values(by="val_f1", ascending=False)

print("\nValidation results after tuning:")
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]["model"]
best_model = best_models[best_model_name]

print("\nSelected best model:", best_model_name)

# ------------------------------------------------------------
# Part 4: Learning Theory Analysis (19.5)
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 4: LEARNING THEORY ANALYSIS")
print("=" * 80)

# Compare shallow vs deep decision trees for bias/variance illustration
shallow_tree = DecisionTreeClassifier(max_depth=2, random_state=42)
deep_tree = DecisionTreeClassifier(max_depth=None, random_state=42)

shallow_tree.fit(X_train, y_train)
deep_tree.fit(X_train, y_train)

models_bias_variance = {
    "ShallowTree": shallow_tree,
    "DeepTree": deep_tree
}

for name, model in models_bias_variance.items():
    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)
    train_acc = accuracy_score(y_train, train_pred)
    val_acc = accuracy_score(y_val, val_pred)
    train_f1 = f1_score(y_train, train_pred)
    val_f1 = f1_score(y_val, val_pred)

    print(f"\n{name}:")
    print("  Train Accuracy:", round(train_acc, 4))
    print("  Validation Accuracy:", round(val_acc, 4))
    print("  Train F1:", round(train_f1, 4))
    print("  Validation F1:", round(val_f1, 4))
    print("  Generalization gap (F1):", round(train_f1 - val_f1, 4))

# Learning curve for the selected best model
train_sizes, train_scores, val_scores = learning_curve(
    estimator=best_model,
    X=X_train_full,
    y=y_train_full,
    cv=5,
    scoring="f1",
    train_sizes=np.linspace(0.2, 1.0, 5),
    n_jobs=-1
)

print("\nLearning curve (F1) for selected best model:")
for i, size in enumerate(train_sizes):
    print(
        f"Train size={size:4d} | "
        f"Train F1={train_scores[i].mean():.4f} | "
        f"CV F1={val_scores[i].mean():.4f}"
    )

# ------------------------------------------------------------
# Part 5: Model Variety (19.6–19.7)
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 5: MODEL VARIETY")
print("=" * 80)

# Linear model
linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(C=1.0, max_iter=2000, random_state=42))
])
linear_model.fit(X_train, y_train)

# Nonparametric model
nonparametric_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=7, weights="distance"))
])
nonparametric_model.fit(X_train, y_train)

linear_val_pred = linear_model.predict(X_val)
nonparam_val_pred = nonparametric_model.predict(X_val)

print("\nLinear model (Logistic Regression):")
print("Validation Accuracy:", round(accuracy_score(y_val, linear_val_pred), 4))
print("Validation F1:", round(f1_score(y_val, linear_val_pred), 4))

print("\nNonparametric model (kNN):")
print("Validation Accuracy:", round(accuracy_score(y_val, nonparam_val_pred), 4))
print("Validation F1:", round(f1_score(y_val, nonparam_val_pred), 4))

# ------------------------------------------------------------
# Part 6: Ensemble Learning (19.8)
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 6: ENSEMBLE LEARNING")
print("=" * 80)

ensemble_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=2,
    random_state=42
)
ensemble_model.fit(X_train, y_train)
ensemble_val_pred = ensemble_model.predict(X_val)

print("\nEnsemble model (Random Forest):")
print("Validation Accuracy:", round(accuracy_score(y_val, ensemble_val_pred), 4))
print("Validation F1:", round(f1_score(y_val, ensemble_val_pred), 4))

# ------------------------------------------------------------
# Part 7: System Design (19.9)
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 7: SYSTEM DESIGN")
print("=" * 80)

# Final retraining on train+validation using the selected best model
best_model.fit(X_train_full, y_train_full)
test_pred = best_model.predict(X_test)

test_acc = accuracy_score(y_test, test_pred)
test_f1 = f1_score(y_test, test_pred)

print("\nFinal selected model evaluated on held-out test set:")
print("Model:", best_model_name)
print("Test Accuracy:", round(test_acc, 4))
print("Test F1:", round(test_f1, 4))

print("\nClassification report:")
print(classification_report(y_test, test_pred))

print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nDeployment-oriented pipeline summary:")
print("- Data collection: usage, support, billing, and engagement features")
print("- Preprocessing: scaling for distance-based and linear models")
print("- Training: compare baseline, tree, linear, kNN, and ensemble models")
print("- Model selection: tune with cross-validation on training data")
print("- Validation: compare generalization before final selection")
print("- Testing: evaluate once on a held-out test set")
print("- Deployment considerations: feature consistency, monitoring, retraining, drift detection")
print("- Business use: score customers by renewal likelihood for targeted retention actions")

print("\n" + "=" * 80)
print("FINAL MODEL COMPARISON SUMMARY")
print("=" * 80)

summary_rows = []

# Refit comparison models on train_full for fair test comparison
comparison_models = {
    "Baseline": DummyClassifier(strategy="most_frequent"),
    "DecisionTree": best_models["DecisionTree"],
    "LogisticRegression": best_models["LogisticRegression"],
    "KNN": best_models["KNN"],
    "RandomForest": best_models["RandomForest"]
}

for name, model in comparison_models.items():
    model.fit(X_train_full, y_train_full)
    pred = model.predict(X_test)
    summary_rows.append({
        "model": name,
        "test_accuracy": accuracy_score(y_test, pred),
        "test_f1": f1_score(y_test, pred)
    })

summary_df = pd.DataFrame(summary_rows).sort_values(by="test_f1", ascending=False)
print(summary_df.to_string(index=False))
print("=" * 80)

# Build and Evaluate a Complete Machine Learning System

## Goal
The goal of this system is to design, train, evaluate, and compare multiple supervised learning models for a classification task. The system also includes model tuning, analysis of generalization, comparison of model types, and discussion of deployment considerations.

---

## Part 1: Learning Setup (19.1–19.2)

### Supervised learning problem
The supervised learning task is:

- **Input:** customer-related features
- **Output label:** whether the customer renews a subscription (`renewed = 1`) or not (`renewed = 0`)

### Features
The features represent behavioral and account-related variables such as:

- monthly usage
- support tickets
- days since login
- average session time
- plan level score
- discount received
- payment reliability
- feature adoption
- contract length score
- engagement trend

### Labels
The label is binary:

- `1 = renewed`
- `0 = not renewed`

### Baseline model
A baseline supervised model was trained using a **DummyClassifier** that always predicts the most frequent class.

This is important because it provides a minimum reference point. A real learning model should perform better than this baseline to be considered useful.

---

## Part 2: Decision Tree Model (19.3)

A **Decision Tree Classifier** was trained as the first real predictive model.

### Why use a decision tree?
Decision trees are useful because they:

- are easy to interpret
- can capture nonlinear relationships
- make decision rules explicit

### Interpretation
The tree structure shows which features are used for splits and in what order. This helps explain the model’s reasoning.

For example, if the tree splits first on engagement or payment reliability, that suggests those variables are especially important for predicting renewal.

---

## Part 3: Model Selection (19.4)

### Hyperparameter tuning
Model selection was performed using **GridSearchCV** with cross-validation.

Each model type had a tuning grid:

- **Decision Tree:** depth, minimum samples per split, minimum samples per leaf
- **Logistic Regression:** regularization strength
- **kNN:** number of neighbors and weighting scheme
- **Random Forest:** number of trees, depth, and leaf size

### Validation comparison
The models were compared on the validation set using metrics such as:

- accuracy
- F1 score

The selected best model was the one with the strongest validation performance, especially on F1, since classification balance matters.

---

## Part 4: Learning Theory Analysis (19.5)

### Bias vs variance
Bias and variance were analyzed by comparing:

- a **shallow decision tree**
- a **deep decision tree**

A shallow tree typically has:

- higher bias
- lower variance
- lower training performance
- more stable validation performance

A deep tree typically has:

- lower bias
- higher variance
- very high training performance
- greater risk of overfitting

### Overfitting risks
Overfitting occurs when the model fits the training data too closely and fails to generalize well to new data.

This can be detected by looking at the gap between:

- training performance
- validation performance

A large gap suggests the model has memorized patterns rather than learned general structure.

### Learning curve
A learning curve was also computed for the selected best model. This helps analyze:

- whether more data improves performance
- whether the model is underfitting or overfitting
- how training and validation scores behave as the sample size increases

---

## Part 5: Model Variety (19.6–19.7)

Two different model families were trained:

### Linear model
**Logistic Regression**

This is a linear model that assumes the decision boundary can be modeled in a relatively simple way after feature scaling.

Advantages:
- interpretable
- efficient
- often generalizes well

### Nonparametric model
**k-Nearest Neighbors (kNN)**

This is a nonparametric model that makes predictions based on nearby training examples.

Advantages:
- flexible
- can capture complex local patterns

Disadvantages:
- sensitive to scaling
- can overfit if `k` is too small
- may struggle when data is noisy

This comparison helps show how model assumptions affect learning behavior.

---

## Part 6: Ensemble Learning (19.8)

An ensemble model, **Random Forest**, was trained.

### Why use an ensemble?
Random forests combine many decision trees and aggregate their outputs.

Advantages:
- lower variance than a single tree
- strong predictive performance
- robust to noise
- can model nonlinear relationships

The ensemble model was then compared to the single-tree, linear, and nonparametric models.

---

## Part 7: System Design (19.9)

### Full ML pipeline
The complete machine learning pipeline includes:

1. **Problem definition**  
   Predict subscription renewal from customer features.

2. **Data preparation**  
   Create features and labels, split data into training, validation, and test sets.

3. **Baseline evaluation**  
   Train a baseline classifier for comparison.

4. **Model training**  
   Train multiple model families:
   - Decision Tree
   - Logistic Regression
   - kNN
   - Random Forest

5. **Model tuning**  
   Use cross-validation and grid search for hyperparameter selection.

6. **Validation-based selection**  
   Select the best-performing model on validation performance.

7. **Generalization analysis**  
   Compare train vs validation behavior and analyze learning curves.

8. **Final test evaluation**  
   Retrain the selected model on train + validation data and evaluate once on the held-out test set.

### Deployment considerations
When deploying the system, several issues matter:

- feature definitions must remain consistent in production
- preprocessing must match training-time preprocessing
- model performance should be monitored over time
- data drift and concept drift should be tracked
- the model may need periodic retraining
- business thresholds may matter in addition to raw accuracy

For example, if this model is used for customer retention, the prediction can help prioritize which customers should receive intervention offers.

---

## What was demonstrated

This learning system demonstrates:

- supervised learning setup
- baseline comparison
- decision tree training and interpretation
- hyperparameter tuning
- validation-based model selection
- bias-variance analysis
- model variety across linear and nonparametric methods
- ensemble learning
- full-system design thinking

This makes it a complete machine learning workflow rather than just a single model experiment.

## Reflection Questions

### Which model generalized best?
The model that generalized best was the one that achieved the strongest validation performance and then maintained strong test performance on the held-out dataset. In many cases, this was the ensemble model, such as the Random Forest, because it balanced flexibility with better resistance to overfitting than a single deep tree. A model generalized well when it performed consistently across training, validation, and test sets rather than only excelling on the training data.

---

### When did overfitting occur?
Overfitting occurred when a model achieved very high training performance but noticeably worse validation performance. This was most visible with overly complex models, such as a deep decision tree, which could fit the training data too closely. The large gap between train and validation metrics showed that the model was learning noise or highly specific patterns rather than generalizable structure.

---

### How did model complexity affect performance?
Model complexity had a major effect on performance. Simpler models, such as logistic regression or shallow trees, were often more stable and generalized better when the data did not require highly complex boundaries. More complex models, such as deep trees or flexible nonparametric models, could capture richer patterns, but they also had a greater risk of overfitting. The best performance usually came from a model with enough complexity to learn important structure, but not so much that it memorized the training data.

---

### What mattered most in building the full system?
What mattered most was not just training one accurate model, but designing the full pipeline carefully. The most important elements were:
- defining the supervised problem clearly
- using a baseline for comparison
- splitting data correctly into train, validation, and test sets
- tuning hyperparameters systematically
- comparing multiple model types
- checking generalization rather than only training accuracy
- thinking about deployment issues such as preprocessing consistency, monitoring, and retraining

Overall, the strongest system came from combining good model choice with disciplined evaluation and full pipeline design.